# Stack Overflow SQL Analytics

## Purpose

Analyze Stack Overflow user activity, content creation and community engagement using SQL.

## Dataset

The project uses a relational database containing Stack Overflow posts, users, votes and badges from 2008.

## Database Scheme

![Scheme](scheme.png)

## Main Objectives

• Explore the database.

• Analyze user activity.

• Investigate post performance.

• Study community engagement.

• Practice advanced SQL techniques.

## Platform Overview

This section provides an overview of the Stack Overflow platform by examining question popularity, posting activity, and early user engagement.

### Popular Questions

In [ ]:
SELECT COUNT(p.id)
FROM stackoverflow.posts AS p
JOIN stackoverflow.post_types AS pt ON pt.id=p.post_type_id
WHERE pt.type= 'Question'
  AND (p.favorites_count >= 100
   OR p.score > 300);

**Output**

1355

### Insight

A total of **1,355 questions** achieved a high level of popularity by either receiving more than **300 score points** or being added to users' favorites at least **100 times**. This indicates that only a relatively small subset of questions generated exceptional community engagement, highlighting the presence of highly valuable content on the platform.

### Daily Question Activity

In [ ]:
WITH q AS
(SELECT COUNT(p.id) AS pac
FROM stackoverflow.posts AS p
JOIN stackoverflow.post_types AS pt ON pt.id=p.post_type_id
WHERE pt.type= 'Question'
  AND creation_date::date BETWEEN '01-11-2008' AND '18-11-2008'
GROUP BY creation_date::date)

SELECT ROUND(AVG(pac))
FROM q;

**Output**

383

### Insight

Between **November 1 and November 18, 2008**, the platform received an average of **383 new questions per day**. This level of activity reflects a highly engaged community with a continuous flow of new discussions and technical questions.

### Users Receiving Badges on Registration Day

In [ ]:
SELECT COUNT(DISTINCT u.id)
FROM stackoverflow.users AS u
JOIN stackoverflow.badges AS b ON u.id=b.user_id
WHERE u.creation_date::date=b.creation_date::date;

**Output**

7047

### Insight

A total of **7,047 users** received at least one badge on the same day they registered. This suggests that a substantial number of new users became active immediately after joining the platform, earning community recognition through early participation.

## User Analytics

This section analyzes user behavior, engagement, recognition, and profile characteristics using SQL queries and window functions.

### Most Voted Posts by Joel Coehoorn

In [ ]:
SELECT COUNT(DISTINCT p.id)
FROM stackoverflow.users AS u
JOIN stackoverflow.posts AS p ON u.id=p.user_id
JOIN stackoverflow.votes AS v ON p.id=v.post_id
WHERE display_name='Joel Coehoorn'
  AND post_id >= 1;

**Output**

12

### Insight

Joel Coehoorn authored 12 unique posts that received at least one vote. This indicates that multiple contributions from the user attracted attention from the community and generated user interaction through voting.

### Most Active Close Voters

In [ ]:
SELECT u.id,
       COUNT(t.name) as votes
FROM stackoverflow.users AS u
JOIN stackoverflow.votes AS v ON u.id=v.user_id
JOIN stackoverflow.vote_types AS t ON v.vote_type_id=t.id
WHERE t.name = 'Close'
GROUP BY u.id
ORDER BY COUNT(t.name) DESC
LIMIT 10;

**Output**

| id | votes |
|:-----:|:-----:|
| 20646 |	36 |
| 14728 |	36 |
| 27163 | 29 |
| 41158 |	24 |
| 24820 |	23 |
| 9345 |	23 |
| 3241 |	23 |
| 44330 |	20 |
| 38426 |	19 |
| 19074 |	19 |

### Insight

The analysis identifies the 10 users who cast the highest number of Close votes. These users played an important role in community moderation by helping identify questions that were unsuitable or should be closed, reflecting strong participation in maintaining content quality.

### Top Badge Earners

In [ ]:
SELECT user_id,
       COUNT(name) as badges,
       DENSE_RANK() OVER (ORDER BY COUNT(name) DESC) as ranking
FROM stackoverflow.badges
WHERE creation_date::date BETWEEN '15-11-2008' AND '15-12-2008'
GROUP BY user_id
ORDER BY COUNT(name) DESC, user_id
LIMIT 10;

**Output**

|user_id | badges |	ranking
|:-----:|:-----:|:-----:|
| 22656 |	149 |	1 |
| 34509 |	45 |	2 |
| 1288 |	40 |	3 |
| 5190 |	31 |	4 |
| 13913 |	30 |	5 |
| 893 |	28 |	6 |
| 10661 |	28 |	6 |
| 33213 |	25 |	7 |
| 12950 |	23 |	8 |
| 25222 |	20 |	9 |

### Insight

The ranking highlights the users who earned the largest number of badges between November 15 and December 15, 2008. Using the DENSE_RANK() window function allows users with the same number of badges to share the same rank, providing a fair comparison of user achievements during the selected period

### User Segmentation by Profile Views

In [ ]:
SELECT id,
       views,
       CASE
           WHEN views >= 350 THEN 1
           WHEN views < 350 AND views >= 100 THEN 2
           WHEN views < 100 THEN 3
       END as group_number
FROM stackoverflow.users
WHERE views <> 0
  AND location LIKE '%United States%';

|id |	views |	group_number|
|:-----:|:-----:|:-----:|
| 3 |	24396 |	1 |
| 13 |	35414 |	1 |
| 23 |	757 |	1 |
| 25 |	3837 |	1 |
| 36 |	505 |	1 |
| 43 |	394 |	1 |
| 45 |	1971 |	1 |
| 50 |	1616 |	1 |
| 64 |	866 |	1 |
| 67 |	8848 |	1 |

### Insight

Users from the United States were grouped into three categories based on the number of profile views. This segmentation distinguishes highly visible users from moderately and less frequently viewed profiles, providing a simple overview of differences in user popularity within the community.

### Top Users Within Each Segment

In [ ]:
WITH res AS
  (SELECT id,
         views,
         CASE
             WHEN views >= 350 THEN 1
             WHEN views < 350 AND views >= 100 THEN 2
             WHEN views < 100 THEN 3
         END AS group_number
  FROM stackoverflow.users
  WHERE views <> 0
    AND location LIKE '%United States%'),
     max as
  (SELECT *,
         MAX(views) OVER (PARTITION BY group_number) AS maks
  FROM res)
SELECT id,
       views,
       group_number
FROM max
WHERE views=maks
ORDER BY views DESC, id;

|id |	views |	group_number|
|:-----:|:-----:|:-----:|
| 16587 |	62813 |	1 |
| 9094	| 349	| 2 |
| 9585 |	349 |	2 |
| 15079 |	349 |	2 |
| 33437 |	349 |	2 |
| 3469 |	99 |	3 |
| 4829 |	99 |	3 |
| 19006 |	99 |	3 |
| 22732 |	99 |	3 |
| 403434 | 99 |	3 |

### Insight

For each profile-view segment, the query identifies the user with the highest number of profile views. This approach highlights the most visible representative of each category while demonstrating the application of window functions for group-wise comparisons.

### Daily User Growth

In [ ]:
WITH sel AS 
  (SELECT EXTRACT(day FROM creation_date::date) As data,
         COUNT(id) OVER (PARTITION BY creation_date::date) AS users,
         COUNT(id) OVER (ORDER BY creation_date::date) AS sum
  FROM stackoverflow.users
  WHERE creation_date BETWEEN '01-11-2008' AND '01-12-2008')
SELECT DISTINCT *
FROM sel
ORDER BY data;

|data |	users |	sum |
|:-----:|:-----:|:-----:|
| 1 |	34 |	34 |
| 2 |	48 |	82 |
| 3 |	75 |	157 |
| 4 |	192 |	349 |
| 5 |	122 |	471 |
| 6 |	132 |	603 |
| 7 |	104 |	707 |
| 8 |	42 |	749 |
| 9	| 45 |	794 |
| 10 |	93 |	887 |
| 11 |	113 |	1000 |
| 12 |	113 |	1113 |
| 13 |	96 |	1209 |
| 14 |	89 |	1298 |
| 15 |	42 |	1340 |
| 16 |	32 |	1372 |
| 17 |	84 |	1456 |
| 18 |	89 |	1545 |
| 19 |	107 |	1652 |
| 20 |	95 |	1747 |
| 21 |	81 |	1828 |
| 22 | 40 |	1868 |
| 23 |	50 |	1918 |
| 24 |	84 |	2002 |
| 25 |	104 |	2106 |
| 26 |	98 |	2204 |
| 27 |	71 |	2275 |
| 28 |	56 |	2331 |
| 29 |	44 |	2375 |
| 30 |	33 |	2408 |

### Insight

The cumulative registration statistics illustrate how the Stack Overflow user base expanded throughout November 2008. In addition to the number of new users registered each day, the cumulative total provides a clear picture of the platform's continuous growth during its early development.

### Time to First Post

In [ ]:
WITH f1 AS
  (SELECT DISTINCT us.id AS user_id,
          us.creation_date AS creation,
          MIN(p.creation_date) OVER (PARTITION BY user_id) AS first
   FROM stackoverflow.users AS us
   JOIN stackoverflow.posts AS p ON us.id=p.user_id
   WHERE p.creation_date IS NOT NULL)
SELECT user_id,
       first - creation as time
FROM f1;

|user_id |	time |
|:-----:|:-----:|
| 1	| 9:18:29 |
| 2	| 14:37:03 |
| 3	| 3 days, 16:17:09 |
| 4	| 15 days, 5:44:22 |
| 5	| 1 day, 14:57:51 |
| 8 |	0:09:29 |
| 9	| 0:32:42 |
| 11 |	0:00:00 |
| 13 |	1:03:17 |
| 17 |	0:04:58 |

### Insight

The query calculates the time elapsed between each user's registration and the creation of their first post. This metric reflects how quickly users became active after joining the platform and can be used as an indicator of initial user engagement.

## Content Analysis

This section focuses on post performance and the relationship between user recognition and content creation.

### Average Post Score by User

In [ ]:
SELECT title,
       user_id,
       score,
       ROUND(AVG(score) OVER (PARTITION BY user_id))
FROM stackoverflow.posts
WHERE score <> 0
  AND title <> '';

**Output**

| title	| user_id |	score |	round |
|-----|:-----:|:-----:|:-----:|
| Escaping Bracket [ in a CONTAINS() clause? |	1 |	10	| 573 |
| How do I calculate someone's age in C#? |	1	| 1743	| 573 |
| Calculate relative time in C#	| 1 |	1348	| 573 |
| Diagnosing Deadlocks in SQL Server 2005 |	1 |	82	| 573 |
| Wrapping StopWatch timing with a delegate or lambda? |	1 |	92 |	573 |
| Parameterize an SQL IN clause | 1	| 953 |	573 |
| Why doesn't IE7 copy <pre><code> blocks to the clipboard correctly? |	1 |	37	| 573 |
| Practical non-image based CAPTCHA approaches?	| 1	| 318 |	573 |
| Why doesn't SQL Full Text Indexing return results for words containing #? |	2 |	19 |	76 |
| Filling a DataSet or DataTable from a LINQ query result set |	2 |	114 |	76 |

### Insight

The analysis compares the score of each individual post with the author's average post score. This provides additional context for evaluating content quality by showing whether a particular post performed above or below the user's typical contribution level, while excluding posts with zero score or missing titles.

### Posts by Highly Awarded Users

In [ ]:
WITH bad AS 
  (SELECT user_id,
          COUNT(creation_date) AS quantity
   FROM stackoverflow.badges
   GROUP BY user_id),
     bad2 AS
  (SELECT user_id
   FROM bad
   WHERE quantity > 1000)
SELECT title
FROM stackoverflow.posts AS pos
RIGHT OUTER JOIN bad2 ON bad2.user_id=pos.user_id
WHERE title <> '';

**Output**

What's the strangest corner case you've seen in C# or .NET?

What's the hardest or most misunderstood aspect of LINQ?

What are the correct version numbers for C#?

Project management to go with GitHub

### Insight

Only 4 posts with non-empty titles were written by users who earned more than 1,000 badges. This demonstrates that the platform's most recognized contributors produced a relatively small number of titled posts within the available dataset, illustrating the relationship between community recognition and content creation.

## Time Series Analysis

This section explores how Stack Overflow activity evolved over time by analyzing monthly trends in post views, publishing activity, and growth rates.

### Monthly Post Views

In [ ]:
SELECT DATE_TRUNC('month', creation_date::date)::date,
       SUM(views_count)
FROM stackoverflow.posts
WHERE creation_date BETWEEN '01-01-2008' AND '31-12-2008'
GROUP BY DATE_TRUNC('month', creation_date::date)
ORDER BY SUM(views_count) DESC;

**Output**

| month |	sum |
|:-----:|:-----:|
| 2008-09-01	| 452928568 |
| 2008-10-01	| 365400138 |
| 2008-11-01	| 221759651 |
| 2008-12-01	| 197792841 |
| 2008-08-01	| 131367083 |
| 2008-07-01	| 669895 |

### Insight

The analysis summarizes the total number of post views for each month in 2008, allowing periods of peak community attention to be identified. Comparing monthly view counts provides an overview of how user interest evolved during the platform's first year of activity.

### Monthly Posts by Selected Users

In [ ]:
WITH users AS
  (SELECT DISTINCT u.id AS id
   FROM stackoverflow.users AS u
   JOIN stackoverflow.posts AS p ON u.id=p.user_id
   WHERE u.creation_date BETWEEN '01-09-2008' AND '01-10-2008'
     AND p.creation_date BETWEEN '01-12-2008' AND '01-01-2009')
SELECT DATE_TRUNC('month', p.creation_date)::date AS month,
       COUNT(p.creation_date)
FROM stackoverflow.posts AS p
RIGHT OUTER JOIN users ON users.id=p.user_id
GROUP BY DATE_TRUNC('month', p.creation_date)::date
ORDER BY DATE_TRUNC('month', p.creation_date)::date DESC;

| month |	count |
|:-----:|:-----:|
| 2008-12-01 |	17641 |
| 2008-11-01 |	18294 |
| 2008-10-01 |	27171 |
| 2008-09-01 |	24870 |
| 2008-08-01 |	32 |

### Insight

This analysis tracks the monthly posting activity of users who registered in September 2008 and remained active by publishing at least one post in December 2008. The results illustrate how content creation evolved over time for a cohort of users who continued contributing after joining the platform.

### Monthly Growth Rate

In [ ]:
WITH first AS
  (SELECT EXTRACT(month FROM creation_date) AS month_number,
          COUNT(creation_date) AS posts
   FROM stackoverflow.posts
   WHERE creation_date BETWEEN '01-09-2008' AND '31-12-2008'
   GROUP BY EXTRACT(month FROM creation_date))
SELECT *,
       ROUND((100 * CAST(posts AS numeric) / LAG(posts, 1) OVER (ORDER BY month_number DESC) - 100),2) AS montly_change
FROM first;

**Output**

| month_number |	posts |	montly_change |
|:-----:|:-----:|:-----:|
| 9	| 70371	|  |
| 10	| 63102 |	-10.33 |
| 11 |	46975 |	-25.56 |
| 12 |	44592 |	-5.07 |

### Insight

The query calculates the monthly percentage change in the number of published posts between September and December 2008. Using the LAG() window function enables direct comparison with the previous month, making it possible to identify periods of growth or decline in platform activity.

## User Engagement Analysis

This section examines how users interacted with the platform after joining, focusing on early participation and short-term activity patterns.

### Highly Active New Users

In [ ]:
select u.display_name as name,
       count(distinct u.id) as cnt_uniq_id
from stackoverflow.users u
join stackoverflow.posts p on u.id = p.user_id
join stackoverflow.post_types pt on pt.id = p.post_type_id
WHERE DATE_TRUNC('day', p.creation_date) >= DATE_TRUNC('day', u.creation_date)
AND DATE_TRUNC('day', p.creation_date) <= DATE_TRUNC('day', u.creation_date) + INTERVAL '1 month'
AND pt.type = 'Answer'
group by 1
having count(p.id) > 100;

**Output**

| name |	cnt_uniq_id |
|:-----:|:-----:|
| 1800 INFORMATION |	1 |
| Adam Bellaire |	1 |
| Adam Davis |	1 |
| Adam Liss	| 1 |
| aku	| 1 |
| Alan	| 8 |
| Amy B	| 1 |
| anjanb |	1 |
| Ben Hoffstein |	1 |
| Brian	| 15 |

### Insight

The analysis identifies users who posted more than 100 answers during their first month on Stack Overflow. Focusing exclusively on answers highlights users who quickly became active contributors by helping other community members rather than asking questions. The results also show how many unique accounts share the same display name.

### Average Active Days

In [ ]:
SELECT ROUND(SUM(active_days) / COUNT(user_id)) AS avg_active_days
FROM
(SELECT user_id, active_days
FROM
(SELECT creation_date::date, user_id,
       COUNT(user_id) OVER (PARTITION BY user_id) AS active_days
FROM stackoverflow.posts
WHERE creation_date::date BETWEEN '2008-12-01' AND '2008-12-07'
GROUP BY 2, 1) AS sum_days
GROUP BY 1, 2) AS uniq_id

**Output**

2

### Insight

Between December 1 and December 7, 2008, users were active on the platform for an average of 2 days, considering only days on which they published at least one post. This metric provides a simple measure of short-term user engagement and posting frequency during the selected period.

## Advanced Window Functions

This section demonstrates the use of SQL window functions for cumulative calculations, ranking, and sequential analysis of user activity.

### Ranking Vote Types

In [ ]:
SELECT *,
       ROW_NUMBER() OVER (ORDER BY id DESC) AS rank
FROM stackoverflow.vote_types
ORDER BY id;

| id |	name |	rank |
|:-----:|:-----:|:-----:|
| 1 |	AcceptedByOriginator |	15 |
| 2 |	UpMod |	14 |
| 3 |	DownMod |	13 |
| 4	| Offensive |	12 |
| 5	| Favorite |	11 |
| 6	| Close	| 10 |
| 7	| Reopen |	9 |
| 8	| BountyStart |	8 |
| 9	| BountyClose |	7 |
| 10 |	Deletion |	6 |
| 11 |	Undeletion |	5 |
| 12 |	Spam |	4 |
| 13 |	InformModerator |	3 |
| 14 |	ModeratorReview |	2 |
| 15 |	ApproveEditSuggestion |	1 |

### Insight

The query assigns a reverse ranking to each vote type using the ROW_NUMBER() window function while preserving the original ordering of the table. Although the transformation itself is simple, it demonstrates the use of window functions to generate sequential rankings independently of the final result ordering.

### Cumulative Post Views

In [ ]:
SELECT user_id,
       creation_date,
       views_count,
       SUM(views_count) OVER (PARTITION BY user_id ORDER BY creation_date)
FROM stackoverflow.posts;

**Output**

| user_id |	creation_date |	views_count |	sum |
|:-----:|:-----:|:-----:|:-----:|
| 1	| 2008-07-31 23:41:00 |	480476|	480476 |
| 1	| 2008-07-31 23:55:38 |	136033 |	616509 |
| 1	| 2008-07-31 23:56:41 |	0 |	616509 |
| 1	| 2008-08-04 02:45:08 |	0 |	616509 |
| 1	| 2008-08-04 04:31:03 |	0 |	616509 |
| 1	| 2008-08-04 08:04:42 |	0 |	616509 |
| 1	| 2008-08-10 08:28:52 |	0 |	616509 |
| 1	| 2008-08-11 19:23:47 |	0 |	616509 |
| 1 | 2008-08-12 00:30:43 |	0 |	616509 |
| 1	| 2008-08-12 04:59:35 |	72431 |	688940 |

### Insight

The query calculates the cumulative number of post views for each user by ordering their posts chronologically and applying the SUM() OVER window function. Running totals provide a clear view of how an author's overall visibility grows over time and represent a common analytical technique used in reporting and dashboard development.

### Weekly Activity of the Most Active User

In [ ]:
WITH a as (SELECT p.user_id as user_id,
COUNT(p.id) as post_cnt
FROM stackoverflow.posts p
GROUP BY user_id
ORDER BY post_cnt DESC
LIMIT 1),

b as (SELECT EXTRACT(WEEK FROM CAST(p.creation_date as date)) as week,
p.creation_date as dt,
ROW_NUMBER() OVER (PARTITION BY EXTRACT(WEEK FROM CAST(p.creation_date as date)) ORDER BY p.creation_date DESC) as rank
FROM stackoverflow.posts p
WHERE p.user_id IN (SELECT user_id
FROM a ) AND p.creation_date::date >= '01-10-2008' AND p.creation_date::date <='31-10-2008')
SELECT DISTINCT week,
dt
FROM b
WHERE rank=1
ORDER BY week

| week |	dt |
|:-----:|:-----:|
| 40 |	2008-10-05 09:00:58 |
| 41 |	2008-10-12 21:22:23 |
| 42 |	2008-10-19 06:49:30 |
| 43 |	2008-10-26 21:44:36 |
| 44 |	2008-10-31 22:16:01 |

### Insight

The analysis identifies the most recent post published during each week of October 2008 by the platform's most active user. Using the ROW_NUMBER() window function makes it possible to rank posts within each week and select only the latest contribution, demonstrating a practical application of partitioning and ranking for time-based analysis.

# Final Conclusions

This project demonstrates the application of SQL for exploratory data analysis using a relational Stack Overflow database. Throughout the analysis, multiple aspects of the platform were investigated, including user behavior, content performance, engagement patterns, and temporal trends.

The project showcases a broad range of SQL techniques commonly required for Data Analyst roles, including:

- filtering and aggregation (WHERE, GROUP BY, HAVING);
- multi-table joins;
- Common Table Expressions (WITH);
- conditional expressions (CASE);
- window functions (ROW_NUMBER, DENSE_RANK, LAG, SUM OVER, AVG OVER, COUNT OVER);
- cumulative calculations;
- ranking and segmentation;
- cohort and time-series analysis.

By combining these techniques with business-oriented interpretations of the results, the project demonstrates not only SQL proficiency but also the ability to transform raw query outputs into meaningful analytical insights.